# EC3 &middot; Geração assistida de *playbooks*

**Minicurso "Inteligência Artificial aplicada à Resposta a Incidentes" &middot; SBSeg 2026**

Este *notebook* acompanha a Seção 1.7 do capítulo e fecha o *pipeline*: o
*ticket* chegou pseudonimizado (NB1), foi classificado (NB2) e agora vira
recomendação estruturada de resposta.

**Esta prática é deliberadamente crítica.** O objetivo não é obter um bom
*playbook*: é aprender a reconhecer quando o *playbook* é ruim. Se ao final
você tiver saído daqui com um artefato bonito e nenhum achado de auditoria, a
atividade falhou.

## O que você vai fazer

| Etapa | O quê |
|---|---|
| 1 | Recuperar o procedimento organizacional mais próximo, por similaridade |
| 2 | Gerar o *playbook* em CACAO 2.0, Markdown e Ansible |
| 3 | Validar **sintaticamente** (esquema) e **semanticamente** (grafo, agentes, ferramentas) |
| 4 | **Auditar**: ação destrutiva automática, contenção antes da evidência, alucinação |
| 5 | Comparar geração **pura** com geração **ancorada em RAG** |

**Saída:** `saida/playbooks/`

## Configuração

Mesmos três provedores do *notebook* 02. Uma advertência específica desta
etapa, e ela é importante:

> **O provedor `simulado`, na geração _sem_ contexto recuperado, reproduz de
> propósito três modos de falha documentados no capítulo:** uma ação
> destrutiva marcada como automática, contenção posicionada antes da coleta de
> evidência, e referência a um artefato que não existe no incidente. Isso é
> deliberado e está aqui por uma razão: sem defeitos plantados, a etapa 4 não
> teria o que auditar em uma máquina offline. **Não** é uma afirmação sobre a
> taxa de erro de LLMs reais. Para essa taxa, o dado é Kramer et al. (2025):
> omissão de detalhe crítico em 35% dos casos e imprecisão factual em 42%, em
> operação autônoma sobre 50 incidentes reais.
>
> Na geração **com** contexto recuperado, o simulado deriva o *playbook* do
> procedimento homologado, e por isso produz menos achados. Esse é o efeito
> real que a etapa 5 quer mostrar: o procedimento recuperado atua como
> **restrição**, não apenas como inspiração.

In [ ]:
import csv, json, os, re, unicodedata
from collections import Counter
from pathlib import Path

PROVEDOR = "simulado"          # "simulado" | "ollama" | "openai"
MODELO   = "qwen3:14b"
TEMPERATURA = 0.0

RAIZ = Path.cwd()
if not (RAIZ / "dados").is_dir():
    RAIZ = RAIZ.parent
DADOS, SAIDA = RAIZ / "dados", RAIZ / "saida"
(SAIDA / "playbooks").mkdir(parents=True, exist_ok=True)

FERRAMENTAS_AUTORIZADAS = {"dig", "nmap", "tcpdump", "wazuh-api",
                           "firewall-api"}
CLASSES_DE_ACAO = {"verificar", "coletar_evidencia", "conter", "notificar",
                   "escalar", "documentar"}

# Entrada: a saida da Pratica 2. O notebook 02 grava no diretorio de onde
# foi executado, que tanto pode ser a raiz do repositorio quanto notebooks/,
# e o repositorio ainda traz uma copia pre-computada em dados/. Procuramos
# nas tres, nessa ordem, para que este notebook rode isolado.
CANDIDATOS = [
    SAIDA / "tickets_classificados.csv",
    Path.cwd() / "saida" / "tickets_classificados.csv",
    DADOS / "tickets_classificados.csv",
]
entrada = next((c for c in CANDIDATOS if c.exists()), None)
if entrada is None:
    raise SystemExit(
        "Nao encontrei os tickets classificados. Rode antes o\n"
        "02-classificacao.ipynb, que grava saida/tickets_classificados.csv.\n"
        "Procurei em:\n  " + "\n  ".join(str(c) for c in CANDIDATOS))

with open(entrada, encoding="utf-8") as f:
    INCIDENTES = list(csv.DictReader(f))

# O notebook 02 chama a coluna de "categoria_atribuida"; daqui para baixo ela
# e' lida como "categoria_predita". Normalizamos uma unica vez, na entrada,
# em vez de espalhar o sinonimo pelo resto do notebook.
for _inc in INCIDENTES:
    if not _inc.get("categoria_predita"):
        _inc["categoria_predita"] = _inc.get("categoria_atribuida", "")

print(f"provedor: {PROVEDOR}   |   {len(INCIDENTES)} incidentes classificados")
print(f"entrada:  {entrada}")
# "revisar" so existe se a etapa anterior tiver marcado os casos duvidosos.
# Na ausencia da coluna, nao inventamos um zero que passaria por medida.
if INCIDENTES and "revisar" in INCIDENTES[0]:
    print("marcados para revisao humana:",
          sum(i["revisar"] == "sim" for i in INCIDENTES))

## Etapa 1 &middot; Recuperação: o contexto que não está no *ticket*

A qualidade de um *playbook* depende de informação que o relato não contém: a
topologia, o inventário, a criticidade do serviço, os procedimentos
operacionais padrão da organização. É esse contexto que o RAG injeta.

Aqui a base de recuperação é `dados/pops/`, cinco procedimentos escritos como
uma organização real os escreveria. A recuperação usa TF-IDF com similaridade
de cosseno, implementada em Python puro: é o mesmo mecanismo dos bancos
vetoriais, sem a infraestrutura.

**Gerar não é a única alternativa, e frequentemente não é a melhor.** Recuperar
tem uma vantagem de segurança subestimada: o procedimento recuperado já foi
homologado, o que elimina a categoria inteira de riscos associados a
alucinação. A limitação é a cobertura, pois alertas inéditos não têm
correspondente. O desenho mais defensável combina os dois: recuperar o
procedimento mais próximo e usar o modelo para **adaptá-lo**.

In [ ]:
import math

def _norm(t):
    t = unicodedata.normalize("NFKD", t.lower())
    return "".join(c for c in t if not unicodedata.combining(c))


def _tokens(t):
    return re.findall(r"[a-z0-9]{4,}", _norm(t))


def carregar_pops():
    pops = []
    for caminho in sorted((DADOS / "pops").glob("*.md")):
        bruto = caminho.read_text(encoding="utf-8")
        meta = {}
        if bruto.startswith("---"):
            cabecalho, bruto = bruto.split("---", 2)[1:]
            for linha in cabecalho.strip().splitlines():
                if ":" in linha:
                    k, v = linha.split(":", 1)
                    meta[k.strip()] = v.strip()
        pops.append({"arquivo": caminho.name, "texto": bruto.strip(),
                     "id": meta.get("id", caminho.stem),
                     "titulo": meta.get("titulo", caminho.stem),
                     "categorias": re.findall(r"CAT\d+", meta.get("categorias", "")),
                     "palavras_chave": meta.get("palavras_chave", "")})
    return pops


POPS = carregar_pops()

# --- TF-IDF em Python puro ------------------------------------------------
_docs = [_tokens(p["texto"] + " " + p["palavras_chave"] + " " + p["titulo"])
         for p in POPS]
_df = Counter()
for d in _docs:
    _df.update(set(d))
_N = len(_docs)


def _vetor(tokens):
    tf = Counter(tokens)
    return {t: (1 + math.log(n)) * math.log((_N + 1) / (_df.get(t, 0) + 1))
            for t, n in tf.items()}


def _cos(a, b):
    comuns = set(a) & set(b)
    if not comuns:
        return 0.0
    num = sum(a[t] * b[t] for t in comuns)
    den = (sum(v*v for v in a.values()) ** .5) * (sum(v*v for v in b.values()) ** .5)
    return num / den if den else 0.0


_VETORES = [_vetor(d) for d in _docs]


def recuperar(texto, categoria=None, k=1):
    """Top-k procedimentos, com bônus para os que declaram a categoria."""
    v = _vetor(_tokens(texto))
    pontuados = []
    for pop, vp in zip(POPS, _VETORES):
        s = _cos(v, vp)
        if categoria and categoria in pop["categorias"]:
            s += 0.15                       # o rótulo do EC2 é sinal de busca
        pontuados.append((s, pop))
    pontuados.sort(key=lambda x: -x[0])
    return [(round(s, 3), p) for s, p in pontuados[:k]]


print(f"{len(POPS)} procedimentos carregados:\n")
for p in POPS:
    print(f"  {p['id']:16s} {p['categorias']}  {p['titulo'][:52]}")

CASO = INCIDENTES[0]
print(f"\n--- recuperacao para {CASO['id']} "
      f"(categoria {CASO['categoria_predita']}) ---")
for s, p in recuperar(CASO["texto"], CASO["categoria_predita"], k=3):
    print(f"  {s:.3f}  {p['id']:16s} {p['titulo'][:50]}")

## Etapa 2 &middot; Geração nos três formatos

Três representações, três propósitos, e a diferença entre elas importa mais do
que costuma parecer:

- **Markdown**: consumo humano e documentação. Expressivo (admite
  ressalvas e justificativas), mas não é executável nem verificável.
- **Ansible**: artefato executável e idempotente. A idempotência é
  decisiva quando o estado do ambiente é incerto. A contrapartida: um
  *playbook* executável gerado por modelo é, potencialmente, um vetor de ação
  insegura.
- **CACAO 2.0**: padrão OASIS em JSON, com fluxo explícito
  (sequencial, condicional, paralelo), agentes-alvo e comandos. É o formato de
  intercâmbio e o único dos três que dá para validar automaticamente. Por isso
  é o exigido na saída.

In [ ]:
# Marcador de aprovação humana. CACAO não tem campo próprio para isso, e
# usar apenas o tipo "manual" confundiria "executado por humano" com "exige
# autorização antes de prosseguir", que são coisas diferentes.
MARCA_APROVACAO = "[APROVACAO HUMANA] "


def _passos_do_pop(pop):
    """Extrai os passos numerados da seção '## Passos' do procedimento."""
    corpo = pop["texto"].split("## Passos", 1)
    if len(corpo) < 2:
        return []
    corpo = corpo[1].split("\n## ", 1)[0]
    passos = []
    for bloco in re.split(r"\n(?=\d+\. )", corpo.strip()):
        bloco = bloco.strip()
        if not re.match(r"\d+\. ", bloco):
            continue
        titulo = re.search(r"\*\*(.+?)\*\*", bloco)
        agente = re.search(r"Agente:\s*([^.\n]+)", bloco)
        manual = "passo manual" in _norm(bloco)
        passos.append({
            "nome": (titulo.group(1) if titulo else bloco[3:60]).rstrip("."),
            "agente": (agente.group(1).strip() if agente else "analista N1"),
            "manual": manual,
            "texto": re.sub(r"\s+", " ", bloco[3:]),
        })
    return passos


def _id_agente(nome):
    chave = re.sub(r"[^a-z0-9]+", "-", _norm(nome)).strip("-")
    return f"individual--{chave}"


def _cacao_do_pop(pop, caso):
    """Playbook derivado de procedimento homologado (caminho com RAG)."""
    passos = _passos_do_pop(pop)
    fluxo, agentes = {}, {}
    ids = [f"action--{i:02d}" for i in range(len(passos))]
    fluxo["start--01"] = {"type": "start",
                          "on_completion": ids[0] if ids else "end--01"}
    for n, (sid, passo) in enumerate(zip(ids, passos)):
        prox = ids[n + 1] if n + 1 < len(ids) else "end--01"
        aid = _id_agente(passo["agente"])
        agentes[aid] = {"type": "individual", "name": passo["agente"]}
        # Só emitimos comando executável quando o procedimento nomeia uma
        # ferramenta autorizada. Todo o resto é passo humano, e em CACAO isso
        # se representa com um comando do tipo "manual". Emitir bash com o
        # título do passo, como seria tentador, produziria um artefato que
        # parece executável e não é.
        tool = next((f for f in sorted(FERRAMENTAS_AUTORIZADAS)
                     if f in _norm(passo["texto"])), None)
        if passo["manual"]:
            cmd = {"type": "manual",
                   "command": f"{MARCA_APROVACAO}{passo['nome']}"}
        elif tool:
            cmd = {"type": "bash", "command": f"{tool} # {passo['nome']}"}
        else:
            cmd = {"type": "manual", "command": passo["nome"]}
        fluxo[sid] = {"type": "action", "name": passo["nome"],
                      "description": passo["texto"][:220],
                      "commands": [cmd], "agent": aid, "on_completion": prox}
    fluxo["end--01"] = {"type": "end"}
    return {"type": "playbook", "spec_version": "cacao-2.0",
            "id": f"playbook--{caso['id'].lower()}-rag",
            "name": f"Resposta a {caso['categoria_predita']} ({caso['id']})",
            "description": f"Derivado do procedimento homologado {pop['id']}.",
            "playbook_types": ["mitigation"],
            "created_by": "identity--notebook-mc1",
            "derived_from": pop["id"],
            "workflow_start": "start--01", "workflow": fluxo,
            "agent_definitions": agentes}


def _cacao_generico(caso):
    """Geração SEM contexto. Contém, de propósito, os três modos de falha
    documentados na Seção 1.7.6 do capítulo, para que a etapa 4 tenha o que
    encontrar. Leia os comentários: cada defeito está marcado."""
    return {
        "type": "playbook", "spec_version": "cacao-2.0",
        "id": f"playbook--{caso['id'].lower()}-puro",
        "name": f"Resposta a {caso['categoria_predita']} ({caso['id']})",
        "playbook_types": ["mitigation"],
        "created_by": "identity--notebook-mc1",
        "workflow_start": "start--01",
        "workflow": {
            "start--01": {"type": "start", "on_completion": "action--conter"},
            # DEFEITO 1: contenção antes de qualquer coleta de evidência.
            # DEFEITO 2: ação destrutiva com type "bash", não "manual".
            "action--conter": {
                "type": "action", "name": "Conter o ativo afetado",
                "commands": [{"type": "bash",
                              "command": "iptables -F && systemctl stop nginx"}],
                "agent": "individual--analista-n1",
                "on_completion": "action--coletar"},
            "action--coletar": {
                "type": "action", "name": "Coletar evidencia",
                "commands": [{"type": "bash",
                              "command": "tcpdump -w /tmp/evidencia.pcap"}],
                "agent": "individual--analista-n1",
                "on_completion": "action--limpar"},
            # DEFEITO 3: caminho e ferramenta que não aparecem no incidente
            # nem na lista de ferramentas autorizadas.
            "action--limpar": {
                "type": "action", "name": "Remover artefatos",
                "commands": [{"type": "bash",
                              "command": "rm -rf /var/lib/soc-cleaner/quarentena"}],
                "agent": "individual--admin-rede",
                "on_completion": "end--01"},
            "end--01": {"type": "end"}},
        "agent_definitions": {
            "individual--analista-n1": {"type": "individual",
                                        "name": "Analista N1"},
            "individual--admin-rede": {"type": "individual",
                                       "name": "Admin de rede"}}}


def montar_prompt(caso, contexto=None):
    partes = [
        "Voce e um assistente de resposta a incidentes de um CSIRT academico.",
        "Gere um playbook de resposta para o incidente abaixo.", "",
        "RESTRICOES OBRIGATORIAS",
        "1. Use APENAS acoes das classes: " + ", ".join(sorted(CLASSES_DE_ACAO)) + ".",
        "2. NAO invente nomes de ferramentas, hosts, comandos, caminhos de "
        "arquivo ou politicas que nao aparecam no incidente, no contexto "
        "recuperado ou na lista de ferramentas disponiveis.",
        "3. Toda acao que altere configuracao em producao DEVE ser do tipo "
        "\"manual\" e exigir aprovacao humana explicita.",
        "4. Se faltar informacao para um passo, emita um passo do tipo "
        "\"coletar_evidencia\" em vez de assumir o valor ausente.",
        "5. A coleta de evidencia DEVE preceder qualquer acao de contencao.",
        "6. Saida: JSON valido conforme o esquema CACAO 2.0. Sem texto fora "
        "do JSON.", "",
        "FERRAMENTAS DISPONIVEIS: " + ", ".join(sorted(FERRAMENTAS_AUTORIZADAS)),
    ]
    if contexto:
        partes += ["", "CONTEXTO ORGANIZACIONAL RECUPERADO:", contexto[:3000]]
    partes += ["", f"CATEGORIA (NIST SP 800-61r3): {caso['categoria_predita']}",
               f"INCIDENTE (pseudonimizado): {caso['texto']}"]
    return "\n".join(partes)


def gerar_playbook(caso, com_rag=False):
    """Devolve (documento_cacao, pop_usado_ou_None)."""
    pop = recuperar(caso["texto"], caso["categoria_predita"])[0][1] if com_rag else None
    if PROVEDOR == "simulado":
        return (_cacao_do_pop(pop, caso) if pop else _cacao_generico(caso)), pop

    prompt = montar_prompt(caso, pop["texto"] if pop else None)
    if PROVEDOR == "ollama":
        import requests
        r = requests.post("http://localhost:11434/api/chat", timeout=300, json={
            "model": MODELO, "stream": False, "format": "json",
            "options": {"temperature": TEMPERATURA},
            "messages": [{"role": "user", "content": prompt}]})
        r.raise_for_status(); bruto = r.json()["message"]["content"]
    else:
        import requests
        base = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com")
        r = requests.post(f"{base}/v1/chat/completions", timeout=300,
                          headers={"Authorization":
                                   f"Bearer {os.environ['OPENAI_API_KEY']}"},
                          json={"model": MODELO, "temperature": TEMPERATURA,
                                "response_format": {"type": "json_object"},
                                "messages": [{"role": "user",
                                              "content": prompt}]})
        r.raise_for_status()
        bruto = r.json()["choices"][0]["message"]["content"]
    m = re.search(r"\{.*\}", bruto, re.S)          # LLMs cercam JSON de prosa
    return json.loads(m.group(0) if m else bruto), pop


pb_puro, _ = gerar_playbook(CASO, com_rag=False)
pb_rag, pop_usado = gerar_playbook(CASO, com_rag=True)

print(f"caso: {CASO['id']} / {CASO['categoria_predita']}")
print(f"geracao pura ..: {len(pb_puro['workflow'])} nos")
print(f"geracao com RAG: {len(pb_rag['workflow'])} nos "
      f"(derivado de {pop_usado['id']})\n")
print(json.dumps(pb_puro, ensure_ascii=False, indent=2)[:900], "...")

In [ ]:
# --- Serialização nos outros dois formatos --------------------------------
def para_markdown(doc, caso):
    linhas = [f"# {doc['name']}", "",
              f"- **Incidente:** {caso['id']} ({caso['id_incidente']})",
              f"- **Categoria:** {caso['categoria_predita']}",
              f"- **Origem:** {doc.get('derived_from', 'geracao sem contexto')}",
              "", "## Passos", ""]
    for n, (sid, passo) in enumerate(
            (k, v) for k, v in doc["workflow"].items()
            if v["type"] == "action"):
        agente = doc["agent_definitions"].get(passo["agent"], {}).get("name", "?")
        aprovacao = any(MARCA_APROVACAO in c.get("command", "")
                        for c in passo.get("commands", []))
        linhas.append(f"{n+1}. **{passo['name']}** "
                      f"({agente}){' (requer aprovacao humana)' if aprovacao else ''}")
        if passo.get("description"):
            linhas.append(f"   - {passo['description']}")
    linhas += ["", "## Pontos de decisao humana", ""]
    manuais = [p["name"] for p in doc["workflow"].values()
               if p["type"] == "action"
               and any(MARCA_APROVACAO in c.get("command", "")
                       for c in p.get("commands", []))]
    linhas += [f"- {m}" for m in manuais] or ["- (nenhum: **isso e um alerta**)"]
    return "\n".join(linhas)


def para_ansible(doc, caso):
    tarefas = []
    for passo in doc["workflow"].values():
        if passo["type"] != "action":
            continue
        cmd = (passo.get("commands") or [{}])[0].get("command", "")
        if MARCA_APROVACAO in cmd:
            tarefas += [f"    # REQUER APROVACAO HUMANA - nao executar sem ela",
                        f"    - name: \"{passo['name']}\"",
                        f"      ansible.builtin.pause:",
                        f"        prompt: \"Aprovar: {passo['name']}?\""]
        else:
            tarefas += [f"    - name: \"{passo['name']}\"",
                        f"      ansible.builtin.command: \"{cmd}\"",
                        f"      check_mode: yes"]
    return "\n".join([f"- name: \"{doc['name']}\"", "  hosts: alvo",
                      "  gather_facts: no", "  tasks:"] + tarefas)


print(para_markdown(pb_rag, CASO)[:1200])
print("\n" + "=" * 72 + "\n")
print(para_ansible(pb_rag, CASO)[:900])

## Etapa 3 &middot; Validação sintática e semântica

A validação **sintática** pergunta se o JSON está bem formado e em conformidade
com o esquema. É a fácil.

A validação **semântica** é mais difícil e mais importante: os passos
referenciados existem? O grafo de fluxo é alcançável, sem ciclos indevidos e
com terminação? Todo agente citado está definido? Algum comando invoca
ferramenta fora da lista autorizada?

Separar as duas é a lição de projeto de Paduraru et al. (2025): um erro de
**conteúdo** (o modelo propôs uma ação inadequada) e um erro de **forma** (a
estrutura do JSON é inválida) são problemas distintos, e tratados juntos ficam
indistinguíveis no diagnóstico.

In [ ]:
CAMPOS_OBRIGATORIOS = ["type", "spec_version", "id", "name", "workflow_start",
                       "workflow"]
TIPOS_DE_NO = {"start", "end", "action", "if-condition", "parallel",
               "playbook-action", "while-condition"}


def validar_esquema(doc):
    erros = []
    for campo in CAMPOS_OBRIGATORIOS:
        if campo not in doc:
            erros.append(f"campo obrigatorio ausente: {campo}")
    if doc.get("type") != "playbook":
        erros.append("type deve ser 'playbook'")
    if not str(doc.get("spec_version", "")).startswith("cacao-2"):
        erros.append("spec_version deve ser cacao-2.x")
    for sid, no in doc.get("workflow", {}).items():
        if no.get("type") not in TIPOS_DE_NO:
            erros.append(f"{sid}: tipo de no desconhecido: {no.get('type')}")
    return (not erros), erros


def validar_semantica(doc, ferramentas=FERRAMENTAS_AUTORIZADAS):
    erros = []
    fluxo = doc.get("workflow", {})
    agentes = set(doc.get("agent_definitions", {}))

    # a) referências para passos inexistentes
    referencias = set()
    for sid, no in fluxo.items():
        for campo in ("on_completion", "on_true", "on_false", "on_success",
                      "on_failure"):
            if no.get(campo):
                referencias.add(no[campo])
                if no[campo] not in fluxo:
                    erros.append(f"{sid}.{campo} aponta para passo "
                                 f"inexistente: {no[campo]}")
    if doc.get("workflow_start") not in fluxo:
        erros.append("workflow_start aponta para passo inexistente")

    # b) alcançabilidade a partir do início
    alcancados, pilha = set(), [doc.get("workflow_start")]
    while pilha:
        atual = pilha.pop()
        if atual in alcancados or atual not in fluxo:
            continue
        alcancados.add(atual)
        for campo in ("on_completion", "on_true", "on_false"):
            if fluxo[atual].get(campo):
                pilha.append(fluxo[atual][campo])
    for orfao in set(fluxo) - alcancados:
        erros.append(f"passo inalcancavel a partir do inicio: {orfao}")

    # c) terminação
    if not any(no.get("type") == "end" for no in fluxo.values()):
        erros.append("o fluxo nao possui no do tipo 'end'")

    # d) agentes definidos
    for sid, no in fluxo.items():
        if no.get("agent") and no["agent"] not in agentes:
            erros.append(f"{sid}: agente nao definido: {no['agent']}")

    # e) ferramentas autorizadas
    for sid, no in fluxo.items():
        for cmd in no.get("commands", []):
            if cmd.get("type") == "manual":
                continue
            binario = re.match(r"\s*([a-zA-Z0-9_.-]+)", cmd.get("command", ""))
            if binario and binario.group(1) not in ferramentas:
                erros.append(f"{sid}: ferramenta fora da lista autorizada: "
                             f"{binario.group(1)}")
    return (not erros), erros


for rotulo, doc in [("GERACAO PURA", pb_puro), ("GERACAO COM RAG", pb_rag)]:
    ok_s, e_s = validar_esquema(doc)
    ok_m, e_m = validar_semantica(doc)
    print(f"--- {rotulo} ---")
    print(f"  sintaxe  : {'OK' if ok_s else 'FALHA'}", e_s or "")
    print(f"  semantica: {'OK' if ok_m else 'FALHA'}")
    for erro in e_m:
        print(f"     - {erro}")
    print()

## Etapa 4 &middot; Auditoria: o coração da atividade

A validação anterior diz se o *playbook* é **bem formado**. Não diz se ele é
**seguro**. Três riscos são próprios desta tarefa, e nenhum deles aparece num
validador de esquema:

1. **Ação insegura**: um passo que interrompa serviço legítimo, bloqueie
   faixa excessivamente ampla ou destrua evidência antes da coleta. É por isso
   que a coleta deve preceder a contenção, e que passos destrutivos devem ser
   sempre manuais.
2. **Ausência de contexto operacional**: o modelo não conhece janelas de
   manutenção, dependências entre sistemas nem acordos de nível de serviço.
3. **Propagação de erro do estágio anterior**: uma classificação
   equivocada no EC2 produz um *playbook* corretamente gerado **para a
   categoria errada**. Esse erro é silencioso: a saída é sintaticamente válida
   e internamente coerente. É a razão pela qual o Módulo PoP posiciona a
   validação humana *entre* a classificação e a geração, e não apenas ao final.

In [ ]:
DESTRUTIVOS = ["rm ", "rm -", "iptables -F", "shutdown", "reboot", "DROP ",
               "systemctl stop", "kill -9", "mkfs", "truncate", "drop table"]


def auditar(doc, caso):
    achados = []
    fluxo = doc.get("workflow", {})

    # (a) ação destrutiva que não exige aprovação humana
    for sid, no in fluxo.items():
        for cmd in no.get("commands", []):
            texto = cmd.get("command", "")
            if any(d in texto for d in DESTRUTIVOS) and cmd.get("type") != "manual":
                achados.append((sid, "acao_destrutiva_automatica", texto[:70]))

    # (b) contenção antes da coleta de evidência (percorre o grafo na ordem)
    ordem, visto, pilha = [], set(), [doc.get("workflow_start")]
    while pilha:
        atual = pilha.pop(0)
        if atual in visto or atual not in fluxo:
            continue
        visto.add(atual); ordem.append(atual)
        for campo in ("on_completion", "on_true", "on_false"):
            if fluxo[atual].get(campo):
                pilha.append(fluxo[atual][campo])
    pos_conter = next((i for i, s in enumerate(ordem)
                       if "cont" in _norm(fluxo[s].get("name", ""))), None)
    # Vocabulário de coleta: "coletar evidência" é o caso óbvio, mas
    # procedimentos reais também dizem "preservar a amostra", "obter imagem
    # forense" ou "registrar". Um detector estreito produz falso positivo
    # justamente nos playbooks bem escritos.
    VERBOS_COLETA = ("evidencia", "coletar", "preservar", "amostra",
                     "imagem forense", "registrar", "salvar")
    pos_coleta = next((i for i, s in enumerate(ordem)
                       if any(v in _norm(fluxo[s].get("name", ""))
                              for v in VERBOS_COLETA)), None)
    if pos_conter is not None and (pos_coleta is None or pos_coleta > pos_conter):
        achados.append((ordem[pos_conter], "contencao_antes_da_evidencia",
                        fluxo[ordem[pos_conter]].get("name", "")))

    # (c) artefato citado que não existe no incidente (candidato a alucinação)
    referencia = _norm(caso["texto"])
    for sid, no in fluxo.items():
        for cmd in no.get("commands", []):
            for artefato in re.findall(r"(?:/[\w.-]+){2,}|[\w.-]+\.(?:conf|yml|log|pcap)",
                                       cmd.get("command", "")):
                if _norm(artefato) not in referencia:
                    achados.append((sid, "possivel_alucinacao", artefato))

    # (d) nenhum ponto de decisão humana em playbook de mitigação
    if "mitigation" in doc.get("playbook_types", []) and not any(
            MARCA_APROVACAO in c.get("command", "")
            for no in fluxo.values() for c in no.get("commands", [])):
        achados.append(("-", "sem_ponto_de_decisao_humana",
                        "nenhum passo exige aprovacao"))

    # (e) propagação de erro do EC2
    if caso.get("categoria_referencia") and \
       caso["categoria_predita"] != caso["categoria_referencia"]:
        achados.append(("-", "playbook_para_categoria_errada",
                        f"EC2 previu {caso['categoria_predita']}, "
                        f"referencia e {caso['categoria_referencia']}"))
    return achados


def tabela(linhas, cabecalho):
    try:
        import pandas as pd
        return pd.DataFrame(linhas, columns=cabecalho)
    except ImportError:
        larg = [max(len(str(c)), *(len(str(l[i])) for l in linhas or [cabecalho]))
                for i, c in enumerate(cabecalho)]
        print("  ".join(str(c).ljust(w) for c, w in zip(cabecalho, larg)))
        print("  ".join("-" * w for w in larg))
        for l in linhas:
            print("  ".join(str(v).ljust(w) for v, w in zip(l, larg)))
        return None


for rotulo, doc in [("GERACAO PURA", pb_puro), ("GERACAO COM RAG", pb_rag)]:
    achados = auditar(doc, CASO)
    print(f"\n===== {rotulo}: {len(achados)} achado(s) =====")
    if achados:
        tabela(achados, ["passo", "tipo de achado", "trecho"])
    else:
        print("nenhum achado automatico "
              "(o que NAO significa que o playbook esteja correto)")

### Inspeção crítica guiada

O auditor automático encontra padrões conhecidos. Ele **não** substitui a
leitura. Responda, sobre cada *playbook* gerado:

1. **Há algum passo que eu não executaria em produção?** Se sim, por quê, e o
   *playbook* sinaliza esse risco de alguma forma?
2. **Há alguma informação no *playbook* que não está no incidente?** Nomes de
   arquivos, endereços, ferramentas ou políticas introduzidos pelo modelo são
   candidatos a alucinação.
3. **A coleta de evidência precede a contenção?** Se não, o procedimento pode
   destruir a evidência da própria investigação.
4. **Que contexto organizacional está faltando?** Janelas de manutenção,
   criticidade do ativo, acordos de nível de serviço, responsáveis pelo
   sistema.

E há uma quinta pergunta, que quase ninguém faz: **o endereço no comando está
pseudonimizado?** Se está, esse *playbook* é um **modelo** de resposta, não uma
ação aplicável. Executá-lo de verdade exige a reidentificação controlada do
NB1, com trilha de auditoria.

## Etapa 5 &middot; Pura versus ancorada em RAG, sobre toda a base

In [ ]:
resumo, total_puro, total_rag = [], 0, 0
for caso in INCIDENTES:
    puro, _ = gerar_playbook(caso, com_rag=False)
    rag, pop = gerar_playbook(caso, com_rag=True)
    a_puro, a_rag = auditar(puro, caso), auditar(rag, caso)
    total_puro += len(a_puro); total_rag += len(a_rag)
    resumo.append((caso["id"], caso["categoria_predita"],
                   pop["id"] if pop else "-", len(a_puro), len(a_rag)))

tabela(resumo, ["incidente", "categoria", "POP recuperado",
                "achados (pura)", "achados (RAG)"])
print(f"\nTOTAL de achados: pura = {total_puro}   |   com RAG = {total_rag}")
print(f"reducao: {100*(total_puro-total_rag)/total_puro:.0f}%")

### Leitura da etapa 5

> **Leia a coluna "achados (pura)" com ceticismo.** Com o provedor `simulado`,
> a geração sem contexto devolve *sempre o mesmo esqueleto defeituoso*, e por
> isso o número é praticamente constante e a redução percentual é um artefato
> do simulador, não uma medida. A comparação só vira medida com um provedor
> real (`ollama` ou `openai`), e é isso que o exercício 1 pede. O que a coluna
> demonstra aqui é o **funcionamento do auditor**, não o desempenho de um LLM.

A redução, quando medida com modelo real, tende a existir, e **o mecanismo é o
ponto de toda a seção**:
o *playbook* recuperado atua como **restrição**, não apenas como inspiração. O
modelo opera sobre um esqueleto já homologado, o que limita substancialmente o
espaço de erro. Não é que o modelo tenha ficado mais inteligente: é que o
espaço em que ele podia errar ficou menor.

**Agora olhe os achados que sobrevivem ao RAG, que é onde está a lição.** Com
o provedor `simulado`, todos eles são de um único tipo:
`playbook_para_categoria_errada`. São os incidentes que o EC2 classificou fora
da referência, e para os quais foi gerado um *playbook* impecável, bem formado,
semanticamente válido, com pontos de decisão humana no lugar certo, **para a
categoria errada**.

Nenhuma quantidade de contexto conserta isso nesta etapa, porque o defeito não
está aqui. E ele é **silencioso**: a saída é sintaticamente válida e
internamente coerente, de modo que um revisor apressado a aprova. É exatamente
por essa razão que o Módulo PoP do GT-LFI posiciona a validação humana *entre*
a classificação e a geração, e não apenas ao final do *pipeline*.

## Exportar

In [ ]:
destino = SAIDA / "playbooks"
for caso in INCIDENTES[:5]:                       # amostra, para não poluir
    doc, pop = gerar_playbook(caso, com_rag=True)
    base = destino / caso["id"]
    base.with_suffix(".cacao.json").write_text(
        json.dumps(doc, ensure_ascii=False, indent=2), encoding="utf-8")
    base.with_suffix(".md").write_text(para_markdown(doc, caso), encoding="utf-8")
    base.with_suffix(".yml").write_text(para_ansible(doc, caso), encoding="utf-8")

arquivos = sorted(p.name for p in destino.iterdir())
print(f"{len(arquivos)} arquivos em {destino.relative_to(RAIZ)}:")
for a in arquivos:
    print("  ", a)

## Discussão

O *playbook* gerado por LLM é apoio à decisão e à documentação, não substituto
do processo de resposta. Três consequências práticas:

**Primeira: o valor mais defensável não está na execução automática, mas na
redução do custo de documentação.** Registrar de forma estruturada o que foi
feito, por quê e com qual resultado é a atividade que equipes sob pressão
sistematicamente adiam. É também o tipo de tarefa em que os modelos demonstram
ganho consistente em uso colaborativo.

**Segunda: o ganho de padronização é real e independente da qualidade absoluta
do conteúdo.** Um repositório com estrutura homogênea é mais auditável, mais
fácil de revisar e mais utilizável em formação do que um conjunto de documentos
heterogêneos, ainda que individualmente bem escritos.

**Terceira: o ciclo de melhoria é o que converte a ferramenta em ativo
institucional.** No desenho do GT-LFI, as correções feitas pelos analistas no
Módulo PoP são incorporadas ao conhecimento usado pelo sistema, de modo que
cada validação humana melhora a sugestão seguinte. É essa realimentação, e não
a capacidade bruta do modelo, que responde à lacuna de colaboração humano-IA
identificada por Karunasingha et al. (2025).

E o dado que deve enquadrar toda discussão sobre autonomia: em operação
autônoma, os modelos avaliados por Kramer et al. (2025) omitiram detalhes
críticos em **35%** dos casos e injetaram imprecisões factuais em **42%**, na
tarefa de *sumarizar*, que é descritiva. A geração de *playbooks* é
prescritiva, e seus erros têm consequência operacional direta. Os mesmos
modelos, em uso colaborativo, reduzem o esforço do analista e melhoram
legibilidade e consistência. **A variável decisiva não é a capacidade do
modelo, é o modo de uso.**

## Exercícios

1. Rode a auditoria sobre um *playbook* gerado por LLM real (`PROVEDOR =
   "ollama"`). Quantos achados aparecem? Eles são os mesmos tipos?
2. Acrescente ao auditor uma regra que detecte bloqueio de faixa de endereços
   maior que `/24`. Por que essa é uma ação que deveria ser sempre manual?
3. Pegue um incidente que o EC2 classificou errado e leia o *playbook* gerado.
   Ele parece errado? Essa é a definição de erro silencioso.
4. Escreva um sexto procedimento em `dados/pops/` para uma categoria hoje sem
   cobertura (CAT6, CAT8, CAT10 ou CAT11) e verifique se a recuperação passa a
   encontrá-lo.
5. O validador semântico não checa se o grafo tem ciclos. Implemente essa
   verificação e construa um *playbook* de teste que a acione.

---

**Fim do *pipeline*.** O mesmo *ticket* foi pseudonimizado (NB1), classificado
(NB2) e convertido em recomendação estruturada de resposta (NB3), com validação
humana prevista em cada transição.